# Candidate recovery metric: daily step count

Steps are the most direct wearable proxy for "back to normal function", and the only candidate
here that can be measured both before and after surgery on the same instrument. This notebook
asks first whether the data exist densely enough, on enough of the cohort, at the right times,
for a steps-based outcome to be definable at all, and then builds that outcome per patient and
characterises what a model would be asked to predict.

The decision this feeds is which single outcome the project commits to. Two numbers carry it: the
paired count -- patients with enough pre-operative days to establish a baseline **and** enough
post-operative days to measure a return to it -- and how much of that population still has an
outcome once the outcome is actually computed.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("fitbit_steps")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

# activity (one row per patient-day worn) and steps_outcome (one recovery ratio per
# patient) are defined in srp_common, not here, so the ratio this notebook characterises
# and the one metric_comparison.ipynb correlates against the EHR metrics are the same
# number computed over the same windows.
MIN_WINDOW_DAYS = srp_common.STEPS_MIN_WINDOW_DAYS

CTES = cfg.cohort_ctes(INDEX_RULE, extra=srp_common.steps_outcome_cte(cfg))

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "activity_summary": ["person_id", "date", "steps"],
})

# 2-coverage

### Does the Fitbit activity table have the columns this notebook assumes?

Table and column names are the one thing a notebook cannot infer, and a wrong guess surfaces as
an opaque BigQuery error a long way into a run. This probe reads only `INFORMATION_SCHEMA`, which
is schema metadata rather than participant data.

In [ ]:
activity_columns = cached_gbq(
    srp_common.columns_sql(cfg, ["activity_summary"]), "activity_summary_columns"
)

if activity_columns.empty:
    print("activity_summary not found in the CDR -- check the table name in the "
          "All of Us data dictionary for this release before going further.")
else:
    print(activity_columns.to_string(index=False))

### How many cohort patients have any daily step record at all?

This is the ceiling on every steps-based outcome: a patient with no activity row can never
contribute one, whatever window is chosen. The gap between the cohort count and this count is
pure, irrecoverable loss.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(*) FROM index_surgery)                AS patients_with_index_date,
        (SELECT COUNT(DISTINCT person_id) FROM activity)    AS patients_with_any_steps,
        (SELECT COUNT(*) FROM activity)                     AS step_days
"""

cov = cached_gbq(coverage_sql, "steps_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

print(f"cohort patients              : {cohort_patients:>9,}")
print(f"  ... with an index date     : {int(cov['patients_with_index_date']):>9,}")
print(f"  ... with >=1 step-day      : {int(cov['patients_with_any_steps']):>9,}"
      f"   ({int(cov['patients_with_any_steps']) / cohort_patients:6.1%} of cohort)")
print(f"step-days contributed        : {int(cov['step_days']):>9,}")

### How does coverage vary with distance from the surgery?

Wearable coverage is not flat in time, and the shape matters more than the headline: adherence
typically collapses in exactly the peri-operative days a recovery metric most needs. The
suppressed variant omits days with fewer than the threshold number of patients.

In [ ]:
curve_sql = CTES + """
    SELECT rel_day, COUNT(DISTINCT person_id) AS n_patients
    FROM activity
    WHERE rel_day BETWEEN -180 AND 365
    GROUP BY rel_day
    ORDER BY rel_day
"""

steps_curve = cached_gbq(curve_sql, "steps_coverage_curve")
steps_curve["share_of_cohort"] = steps_curve["n_patients"] / cohort_patients

for label, (lo, hi) in WINDOWS.items():
    window = steps_curve[steps_curve["rel_day"].between(lo, hi)]
    if not window.empty:
        print(f"{label:9s} [{lo:>4}, {hi:>4}]  median patients/day = "
              f"{int(window['n_patients'].median()):>7,}"
              f"   ({window['share_of_cohort'].median():6.1%} of cohort)")

srp_common.plot_rel_day_curve(
    cfg, steps_curve, "n_patients",
    title="Patients with a Fitbit step record, by day relative to surgery",
    ylabel="patients with a step record", suppress=False, save_as="steps_coverage")
srp_common.plot_rel_day_curve(
    cfg, steps_curve, "n_patients",
    title="Patients with a Fitbit step record, by day relative to surgery",
    ylabel="patients with a step record", suppress=True, save_as="steps_coverage")

### How many patients have both a usable baseline and a usable follow-up?

A return-to-baseline outcome needs two windows populated for the same patient, so the binding
constraint is the paired count rather than either window alone. This is the single number that
decides whether a steps-based outcome is viable for this cohort.

In [ ]:
window_case = srp_common.window_case_sql("rel_day")

paired_sql = CTES + f"""
    , per_window AS (
        SELECT
            person_id,
            {window_case} AS window_name,
            COUNT(DISTINCT rel_day) AS n_days
        FROM activity
        WHERE {window_case} IS NOT NULL
        GROUP BY person_id, window_name
    ),
    wide AS (
        SELECT
            person_id,
            MAX(IF(window_name = 'baseline', n_days, 0)) AS baseline_days,
            MAX(IF(window_name = 'postop',   n_days, 0)) AS postop_days
        FROM per_window
        GROUP BY person_id
    )
    SELECT
        COUNT(*)                                                    AS patients_any_window,
        COUNTIF(baseline_days >= 7)                                 AS baseline_7d,
        COUNTIF(postop_days >= 7)                                   AS postop_7d,
        COUNTIF(baseline_days >= 7  AND postop_days >= 7)           AS paired_7d,
        COUNTIF(baseline_days >= 14 AND postop_days >= 14)          AS paired_14d
    FROM wide
"""

paired = cached_gbq(paired_sql, "steps_paired_windows").iloc[0]
paired_7d = int(paired["paired_7d"])

for label in ["patients_any_window", "baseline_7d", "postop_7d",
              "paired_7d", "paired_14d"]:
    n = int(paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:22s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

print()
print("--> paired_7d is the analysable population for a return-to-baseline outcome.")

# 3-signal

### Does the step count actually move after surgery?

A metric that no surgery visibly perturbs cannot be a recovery outcome, however good its
coverage. Quantiles are computed inside BigQuery so only the summary crosses into the notebook.

In [ ]:
signal_sql = CTES + """
    SELECT
        rel_day,
        COUNT(DISTINCT person_id)                    AS n_patients,
        APPROX_QUANTILES(steps, 100)[OFFSET(25)]     AS p25_steps,
        APPROX_QUANTILES(steps, 100)[OFFSET(50)]     AS p50_steps,
        APPROX_QUANTILES(steps, 100)[OFFSET(75)]     AS p75_steps
    FROM activity
    WHERE rel_day BETWEEN -60 AND 180
    GROUP BY rel_day
    ORDER BY rel_day
"""

steps_signal = cached_gbq(signal_sql, "steps_signal_curve")

srp_common.plot_rel_day_curve(
    cfg, steps_signal, "p50_steps", band=("p25_steps", "p75_steps"),
    title="Median daily steps (IQR band), by day relative to surgery",
    ylabel="steps per day", suppress=False, save_as="steps_signal")
srp_common.plot_rel_day_curve(
    cfg, steps_signal, "p50_steps", band=("p25_steps", "p75_steps"),
    title="Median daily steps (IQR band), by day relative to surgery",
    ylabel="steps per day", suppress=True, save_as="steps_signal")

### How long does the median take to come back to baseline?

If the median never returns within the observed horizon, a "days to recovery" outcome is censored
for most patients and a fixed-horizon ratio is the better construction. The 7-day sustained rule
avoids calling a single good day a recovery.

In [ ]:
RECOVERY_FRACTION = 0.95
SUSTAIN_DAYS = 7

base_lo, base_hi = WINDOWS["baseline"]
baseline_level = steps_signal.loc[
    steps_signal["rel_day"].between(base_lo, base_hi), "p50_steps"].median()
target = RECOVERY_FRACTION * baseline_level

post = steps_signal[steps_signal["rel_day"] >= 0].reset_index(drop=True)
at_or_above = (post["p50_steps"] >= target).astype(int)
sustained = at_or_above.rolling(SUSTAIN_DAYS).sum() == SUSTAIN_DAYS

nadir_row = post.loc[post["p50_steps"].idxmin()]
print(f"baseline median steps (days {base_lo}..{base_hi}) : {baseline_level:,.0f}")
print(f"target ({RECOVERY_FRACTION:.0%} of baseline)                 : {target:,.0f}")
print(f"post-operative nadir                        : {nadir_row['p50_steps']:,.0f} "
      f"on day {int(nadir_row['rel_day'])}")

if sustained.any():
    # idxmax gives the row the window ENDS on; the run starts SUSTAIN_DAYS-1 rows earlier.
    # Indexing back by position rather than subtracting days keeps this correct when the
    # curve has gaps, which it will wherever a day fell below the suppression threshold.
    day = int(post.loc[sustained.idxmax() - SUSTAIN_DAYS + 1, "rel_day"])
    print(f"median first sustains the target on day     : {day}")
    days_to_recovery = day
else:
    print(f"median never sustains the target within day {int(post['rel_day'].max())} "
          "-- a time-to-recovery outcome would be censored for most patients")
    days_to_recovery = None

# 4-outcome

### What does the patient-level outcome look like once it is actually built?

The median curve above can return to baseline while half the individual patients never do, so the
distribution of the per-patient ratio is the thing a model would be asked to predict rather than
the curve. Quantiles and histogram buckets are computed inside BigQuery, so only counts cross into
the notebook.

In [ ]:
ratio_sql = CTES + f"""
    SELECT
        COUNT(*)                                                     AS patients_with_activity,
        COUNTIF(steps_recovery_ratio IS NOT NULL)                    AS n_eligible,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(10)] AS p10_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(25)] AS p25_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(50)] AS p50_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(75)] AS p75_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(90)] AS p90_ratio,
        COUNTIF(steps_recovery_ratio < 0.5)                          AS n_under_half,
        COUNTIF(steps_recovery_ratio >= {RECOVERY_FRACTION})         AS n_at_baseline
    FROM steps_outcome
"""

ratio = cached_gbq(ratio_sql, "steps_outcome_ratio").iloc[0]
n_eligible = int(ratio["n_eligible"])

print(f"patients with any step-day     : {int(ratio['patients_with_activity']):>9,}")
print(f"  ... with the ratio defined   : {n_eligible:>9,}"
      f"   ({n_eligible / cohort_patients:6.1%} of cohort)")
print(f"    (needs >= {MIN_WINDOW_DAYS} worn days in each of the baseline and postop windows)")
print()
for label, key in [("p10", "p10_ratio"), ("p25", "p25_ratio"), ("p50", "p50_ratio"),
                   ("p75", "p75_ratio"), ("p90", "p90_ratio")]:
    print(f"  {label} recovery ratio            : {ratio[key]:9.3f}")
print()
print(f"below half their baseline      : {int(ratio['n_under_half']):>9,}"
      f"   ({int(ratio['n_under_half']) / n_eligible:6.1%} of eligible)")
print(f"at or above {RECOVERY_FRACTION:.0%} of baseline    : {int(ratio['n_at_baseline']):>9,}"
      f"   ({int(ratio['n_at_baseline']) / n_eligible:6.1%} of eligible)")

# Bucketed in SQL rather than by binning a per-patient frame in pandas, which would mean
# pulling one row per participant into the notebook. The top bucket is open-ended: a
# patient can be several times their baseline, and a long thin tail of near-empty buckets
# is both unreadable and the part most likely to fall under the threshold.
hist_sql = CTES + """
    SELECT
        LEAST(FLOOR(steps_recovery_ratio * 5) / 5, 2.0) AS ratio_bucket,
        COUNT(*)                                        AS n_patients
    FROM steps_outcome
    WHERE steps_recovery_ratio IS NOT NULL
    GROUP BY ratio_bucket
    ORDER BY ratio_bucket
"""

hist = cached_gbq(hist_sql, "steps_outcome_histogram")
hist["bucket"] = [f"{v:.1f}+" if v >= 2.0 else f"{v:.1f}" for v in hist["ratio_bucket"]]

srp_common.plot_distribution(
    cfg, hist, "bucket",
    title="Steps recovery ratio, one value per patient",
    xlabel="postop median steps / baseline median steps",
    suppress=True, save_as="steps_outcome_ratio")

### How many patients would a time-to-recovery outcome be censored for?

A ratio is defined for everyone with wear on both sides of the surgery, but a days-to-recovery
outcome is only observed for patients who reach their own baseline inside the horizon, and the
rest are censored rather than scored badly. The same target and sustain rule as the median curve
above is applied per patient, so the two answers are about one definition of recovery.

In [ ]:
# The postop window is 30 days because that is what screening the metric needed; a
# time-to-recovery outcome has to look further out or almost everyone is censored by
# construction. This is the re-split that DEFAULT_WINDOWS' own comment anticipates, declared
# here rather than in the shared windows so the screening numbers stay comparable across metrics.
RECOVERY_HORIZON = 180

# A trailing week that contains only one worn day is not evidence of a sustained level, so a
# run has to carry at least this many days before it can count as a recovery.
MIN_DAYS_IN_RUN = 4

RECOVERY_CTE = f"""
    , daily AS (
        SELECT
            a.person_id,
            a.rel_day,
            a.steps,
            o.baseline_steps
        FROM activity AS a
        JOIN steps_outcome AS o ON o.person_id = a.person_id
        WHERE o.steps_recovery_ratio IS NOT NULL
          AND a.rel_day BETWEEN 1 AND {RECOVERY_HORIZON}
    ),
    smoothed AS (
        SELECT
            person_id,
            rel_day,
            baseline_steps,
            AVG(steps) OVER trailing_week AS steps_run,
            COUNT(*)   OVER trailing_week AS days_in_run
        FROM daily
        -- RANGE, not ROWS: the frame has to be the {SUSTAIN_DAYS} calendar days ending on this
        -- one. A row frame would span months for a patient who wore the device on
        -- {SUSTAIN_DAYS} scattered days, and would call that a sustained recovery.
        WINDOW trailing_week AS (
            PARTITION BY person_id ORDER BY rel_day
            RANGE BETWEEN {SUSTAIN_DAYS - 1} PRECEDING AND CURRENT ROW
        )
    ),
    recovered AS (
        SELECT
            person_id,
            MIN(rel_day) AS day_recovered
        FROM smoothed
        WHERE days_in_run >= {MIN_DAYS_IN_RUN}
          AND steps_run >= {RECOVERY_FRACTION} * baseline_steps
        GROUP BY person_id
    )"""

censoring_sql = CTES + RECOVERY_CTE + """
    SELECT
        COUNT(*)                                              AS n_recovered,
        APPROX_QUANTILES(day_recovered, 100)[SAFE_OFFSET(25)] AS p25_day,
        APPROX_QUANTILES(day_recovered, 100)[SAFE_OFFSET(50)] AS p50_day,
        APPROX_QUANTILES(day_recovered, 100)[SAFE_OFFSET(75)] AS p75_day
    FROM recovered
"""

censoring = cached_gbq(censoring_sql, "steps_time_to_recovery").iloc[0]
n_recovered = int(censoring["n_recovered"])
share_recovered = n_recovered / n_eligible
median_days_to_recovery = (int(censoring["p50_day"]) if n_recovered >= SUPPRESS_THRESHOLD
                           else None)

print(f"eligible patients                     : {n_eligible:>9,}")
print(f"  ... reaching {RECOVERY_FRACTION:.0%} of baseline for "
      f"{SUSTAIN_DAYS} days : {n_recovered:>9,}   ({share_recovered:6.1%})")
print(f"  ... censored at day {RECOVERY_HORIZON:<3d}              : "
      f"{n_eligible - n_recovered:>9,}   ({1 - share_recovered:6.1%})")
print()
if n_recovered < SUPPRESS_THRESHOLD:
    print(f"Fewer than {SUPPRESS_THRESHOLD} patients recover within the horizon, so the day "
          "quantiles are not reportable.")
else:
    print(f"days to recovery among those who reach it : "
          f"p25={int(censoring['p25_day'])}  p50={int(censoring['p50_day'])}  "
          f"p75={int(censoring['p75_day'])}")

# Which construction survives this is the point: heavy censoring makes the fixed-horizon
# ratio the tractable outcome and days-to-recovery a survival problem, not a regression one.
if share_recovered < 0.5:
    print(f"\n--> most eligible patients never recover within {RECOVERY_HORIZON} days. A "
          "days-to-recovery outcome would be censored for the majority.")

### Do the two constructions rank the same patients as poor recoverers?

If the ratio and the time to recovery disagree about who did badly, the project is choosing between
two different outcomes rather than two encodings of one, and that choice has to be deliberate. A
tertile cross-tabulation rather than a correlation, because time to recovery is censored for
whoever never reaches baseline and a correlation would have to drop exactly those patients.

In [ ]:
agreement_sql = CTES + RECOVERY_CTE + f"""
    , scored AS (
        SELECT
            o.steps_recovery_ratio,
            -- A censored patient is the slowest recovery observed, not a missing value:
            -- they are known not to have recovered by the horizon, so they sort past
            -- everyone who did.
            COALESCE(r.day_recovered, {RECOVERY_HORIZON} + 1) AS day_recovered
        FROM steps_outcome AS o
        LEFT JOIN recovered AS r ON r.person_id = o.person_id
        WHERE o.steps_recovery_ratio IS NOT NULL
    ),
    tertiles AS (
        SELECT
            NTILE(3) OVER (ORDER BY steps_recovery_ratio)  AS ratio_tertile,
            NTILE(3) OVER (ORDER BY day_recovered DESC)    AS speed_tertile
        FROM scored
    )
    SELECT ratio_tertile, speed_tertile, COUNT(*) AS n_patients
    FROM tertiles
    GROUP BY ratio_tertile, speed_tertile
    ORDER BY ratio_tertile, speed_tertile
"""

agreement = cached_gbq(agreement_sql, "steps_outcome_agreement")
smallest = srp_common.min_cell(agreement, "n_patients")

print(f"smallest cell: {smallest:,}")
if smallest < SUPPRESS_THRESHOLD:
    print(f"At least one cell is below the {SUPPRESS_THRESHOLD}-patient threshold, so this "
          "table may not be reported as-is. Widen the tertiles to halves, or aggregate.")
else:
    table = agreement.pivot(index="ratio_tertile", columns="speed_tertile",
                            values="n_patients")
    print()
    print("rows = recovery ratio tertile, columns = time-to-recovery tertile; 1 = worst in "
          "both, since time to recovery is ordered descending so that a slow or censored "
          "recovery sorts as a poor outcome")
    print(table.to_string())
    on_diagonal = sum(int(table.loc[i, i]) for i in table.index if i in table.columns)
    total = int(agreement["n_patients"].sum())
    print(f"\nagreeing on tertile: {on_diagonal:,} of {total:,} ({on_diagonal / total:.1%})")

# 5-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="activity_summary",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    outcome_definition=(
        f"median daily steps over days {WINDOWS['postop'][0]}..{WINDOWS['postop'][1]} "
        f"divided by median daily steps over days {WINDOWS['baseline'][0]}.."
        f"{WINDOWS['baseline'][1]}, both windows requiring >= {MIN_WINDOW_DAYS} worn days"
    ),
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_steps"]),
    # The analysable population is the one the outcome is actually defined for, which is
    # narrower than paired_7d: a patient can clear both windows and still have a baseline
    # median of zero, leaving the ratio undefined.
    patients_analysable=n_eligible,
    coverage_share=n_eligible / cohort_patients,
    horizon_days=int(WINDOWS["postop"][1]),
    days_to_recovery=days_to_recovery,
    median_recovery_ratio=float(ratio["p50_ratio"]),
    recovery_horizon_days=RECOVERY_HORIZON,
    share_recovered_by_horizon=share_recovered,
    median_days_to_recovery=median_days_to_recovery,
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Non-wear is plausibly informative -- a patient recovering badly is less likely "
           "to wear the device -- so missingness must be treated as censoring, not noise."),
)